In [2]:
"""
Author: Grant Manley
Class: CS340

Description:

Austin Animal Shelter Database Dashboard for Grazioso Salvare
-------------------------------------------------------------
Collection of animals and their information with specialized
filters to sort those with rescue capabilities. Interactive
widgets include map with animal location and pie chart of 
breeds for current selection.

Dependencies:
    - dash
    - dash_leaflet
    - plotly
    - pandas
    - numpy
    - jupyter_dash
    - CRUB_Python_Module

"""


# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html, callback_context, dash
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD module AnimalShelter
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

# User authentication credentials
username = "aacuser"
password = "aacuserpw1"

# Validation check
if not username or not password:
    raise ValueError("Credentials cannot be empty.")
    
# Instantiate shelter CRUD object and pull data set for validation check
shelter = AnimalShelter(username, password)
shelter_data = shelter.read({})

# Validate data set is not empty
if not shelter_data:
    raise RuntimeError("Data set pulled through MongoDB is empty.")

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if '_id' in df.columns:
    df.drop(columns=['_id'],inplace=True)
    
# Validation setup for necessary columns during map rendering
REQUIRED_COLUMNS = {
    'lat': 13,
    'lon': 14,
    'breed': 4,
    'name': 9
}
if len(df.columns) <= max(REQUIRED_COLUMNS.values()):
    raise ValueError(f"Dataset does not have the necessary columns for map rendering.")

# Dictionary for filtering rescue types
RESCUE_TYPES = {
    'water': {
        'animal_type': 'Dog',
        'sex_upon_outcome': 'Intact Female',
        'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
        'breed': {'$in': [
            'Labrador Retriever Mix',
            'Chesapeake Bay Retriever',
            'Newfoundland' ]}
    },
    'mountain_wilderness': {
        'animal_type': 'Dog',
        'sex_upon_outcome': 'Intact Male',
        'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
        'breed': {'$in': [
            'German Shepherd',
            'Alaskan Malamute',
            'Old English Sheepdog',
            'Siberian Husky',
            'Rottweiler' ]}
    },
    'disaster_tracking': {
        'animal_type': 'Dog',
        'sex_upon_outcome': 'Intact Male',
        'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300},
        'breed': {'$in': [
            'Doberman Pinscher',
            'German Shepherd',
            'Golden Retriever',
            'Bloodhound',
            'Rottweiler' ]}
    }
}
        
        
    
#########################
# Dashboard Layout / View
#########################
app = JupyterDash('Austin Animal Center')

# Button off style
button_off = {
    'padding': '8px 16px',
    'marginRight': '10px',
    'backgroundColor': '#f0f0f0',
    'color': '#333',
    'border': '2px solid #aaa',
    'borderRadius': '4px',
    'cursor': 'pointer'
}

# Button on style
button_on = {
    'padding': '8px 16px',
    'marginRight': '10px',
    'backgroundColor': '#4CAF50',
    'color': 'white',
    'border': '2px solid #4CAF50',
    'borderRadius': '4px',
    'cursor': 'pointer'
}

# Button reset style
button_reset = {
    'padding': '8px 16px',
    'marginRight': '10px',
    'backgroundColor': '#f0f0f0',
    'color': '#333',
    'border': '2px solid #aaa',
    'borderRadius': '4px',
    'cursor': 'pointer',
    'marginLeft': 'auto'
}

# Layout
app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    
    # Storage for active button selections
    dcc.Store(id='active-buttons', data=[]),
    
    # Grazioso Salvare's logo
    html.Center(
        html.A(
            html.Img(
                src='assets/Grazioso Salvare Logo.png',
                style={'height': '100px', 'marginBottom': '0px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        )
    ),
    
    # Headers
    html.Center(html.B(html.H1('Austin Animal Center'))),
    html.Center(html.B(html.H2('Author: Grant Manley'))),
    html.Hr(),
    
    # Toggle button on/off look based on clicks
    html.Div([
        html.Button("Cats", id='btn-cat', n_clicks=0, style=button_off),
        html.Button('Dogs', id='btn-dog', n_clicks=0, style=button_off),
        html.Span(style={'display': 'inline-block', 'width': '40px'}),
        html.Button('Water Rescue', id='btn-water',
                    n_clicks=0, style=button_off),
        html.Button('Mountain/Wilderness Rescue', id='btn-mountain_wilderness',
                    n_clicks=0, style=button_off),
        html.Button('Disaster/Tracking Rescue', id='btn-disaster_tracking',
                    n_clicks=0, style=button_off),
        html.Button('Reset', id='btn-reset',
                    n_clicks=0, style=button_reset),
    ], style={'marginBottom': '15px', 'display': 'flex'}),
    
    html.Hr(),
    
    # Datatable populated through CRUD & MongoDB settings
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode='multi',
        column_selectable=False,
        row_selectable='single',
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action='native',
        page_current=0,
        
        # Limits documents displayed per page
        page_size=10,
        
        # Detailed styling for tables
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'maxWidth': '180px',
            'whiteSpace': 'normal',
            'fontSize': '13px',
        },
        style_header={
            'backgroundColor': 'rgb(144, 238, 144)',
            'fontWeight': 'bold',
            'textAlign': 'center',
        },
        style_data={
            'backgroundColor': 'white',
            'color': 'black',
        },
        
        style_data_conditional=[
            {
                # Odd row highlight
                'if': {'row_index': 'odd'},
                'backgroundColor': 'rgb(245, 245, 245)',
            },
            {
                # Row 0 initialization highlight
                'if': {'row_index': 0},
                'backgroundColor': 'rgb(255, 220, 220)',
                'border': '1px solid rgb(255, 100, 100)',
            },
            {
                # Selected row highlight overrides above
                'if': {'state': 'selected'},
                'backgroundColor': 'rgb(173, 216, 230)',
                'border': '1px solid rgb(70, 130, 180)',
            }
        ],
    ),
    html.Br(),
    html.Hr(),
    
    # Size & Styling for piechart and map.
    html.Div([
        html.Div(
            dcc.Graph(id='pie-chart', style={'height': '500px'}),
            style={'display': 'inline-block', 'width': '48%',
                   'verticalAlign': 'top'}
        ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            style={'display': 'inline-block', 'width': '48%',
                   'verticalAlign': 'top'}
        ),
    ], style={'display': 'flex', 'flexDirection': 'row', 'width': '100%', 'gap': '1%', 'justifyContent': 'center'}),
], style={'maxWidth': '1400px', 'margin': '0 auto', 'padding': '0 20px'})

#############################################
# Interaction Between Components / Controller
#############################################

###################################################################################
#
# Callbacks for filter_and_style
#
# Description: Monitors for user clicks and inputs to properly update filters,
# click counts, and stylings.
#
# Inputs:
#  btn-cat n_clicks (int) - Cat filter button click count
#  btn-dog n_clicks (int) - Dog filter button click count
#  btn-water n_clicks (int) - Water rescue dogs button click count
#  btn-mountain_wilderness n_clicks (int) - Mountain wilderness rescue dogs button click count
#  btn-disaster_tracking n_clicks (int) - Disaster tracking rescue dogs button click count
#  btn-reset n_clicks (int) - Reset button click count
#  selected_columns (list) - Column ID selections
#  derived_virtual_selected_rows (list) - Selected rows
#
# State:
#  active-buttons data (list) - Current ON button's IDs
#
# Outputs:
#  datatable-id data (list) - Filtered records
#  datatable-id style_data_conditional (list) - Highlight styling rules for rows/columns
#  datatable-id filter_query (str) - no_update/cleared when reset
#  active-buttons data (list) - Buttons that are currently ON
#  btn-cat style (dict) - on/off styling
#  btn-dog style (dict) - on/off styling
#  btn-water style (dict) - on/off styling
#  btn-mountain_wilderness style (dict) - on/off styling
#  btn-disaster_tracking style (dict) - on/off styling
#  btn-reset style (dict) - Always button_reset to ensure proper spacing
#
###################################################################################

@app.callback(
    [Output('datatable-id', 'data'),
     Output('datatable-id', 'style_data_conditional'),
     Output('datatable-id', 'filter_query'),
     Output('active-buttons', 'data'),
     Output('btn-cat', 'style'),
     Output('btn-dog', 'style'),
     Output('btn-water', 'style'),
     Output('btn-mountain_wilderness', 'style'),
     Output('btn-disaster_tracking', 'style'),
     Output('btn-reset', 'style')],
    [Input('btn-cat', 'n_clicks'),
     Input('btn-dog', 'n_clicks'),
     Input('btn-water', 'n_clicks'),
     Input('btn-mountain_wilderness', 'n_clicks'),
     Input('btn-disaster_tracking', 'n_clicks'),
     Input('btn-reset', 'n_clicks'),
     Input('datatable-id', 'selected_columns'),
     Input('datatable-id', 'derived_virtual_selected_rows')],
    [State('active-buttons', 'data')]
)

##############################################################################
#
# Description: Filters and styles the two buttons at the top responsible for 
# selecting only cats or dogs.
# 
# Param: cat_clicks (int) - # of times the cat button has been clicked
# Param: dog_clicks (int) - # of times the dog button has been clicked
# Param: water_clicks (int) - # of times the water button has been clicked
# Param: mountain_wilderness_clicks (int) - # of times the mountain wilderness button has been clicked
# Param: disaster_tracking_clicks (int) - # of times the disaster tracking button has been clicked
# Param: reset_clicks (int) - track if reset button has been clicked
# Param: selected_columns (list) - Column IDs selected in table
# Param: selected_rows (list) - Currently selected row indices
# Param: active_buttons (list) - IDs of buttons currently ON
# 
# Returns: tuple - list: Filtered records from table
#                  list: Column stylings
#                  str: filter_query no_update or reset to ''
#                  list: Buttons currently ON
#                  dict: Styling for cat button on/off
#                  dict: Styling for dog button on/off
#                  dict: Styling for water button on/off
#                  dict: Styling for mountain_wilderness button on/off
#                  dict: Styling for disaster_tracking button on/off
#                  dict: Styilng for reset button to maintain spacing
#
##############################################################################

def filter_and_style(cat_clicks,
                     dog_clicks,
                     water_clicks,
                     mountain_wilderness_clicks,
                     disaster_tracking_clicks,
                     reset_clicks,
                     selected_columns,
                     selected_rows, active_buttons):
    
    ctx = callback_context
    trigger_id = ctx.triggered[0]['prop_id'].split('.')[0] if ctx.triggered else None
    is_reset  = trigger_id == 'btn-reset'
    is_button = trigger_id in ('btn-cat', 'btn-dog', 'btn-water',
                               'btn-mountain_wilderness', 'btn-disaster_tracking', 'btn-reset')
    
    # Highlight for selected columns
    column_style = [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in (selected_columns or [])]
    
    # Alternating row striping for visibility
    column_style += [
        {
            'if': {'row_index': 'odd'},
            'backgroundColor': 'rgb(245, 245, 245)',
        },
        {
            'if': {'state': 'selected'},
            'backgroundColor': 'rgb(173, 216, 230)',
            'border': '1px solid rgb(70, 130, 180)',
        }
    ]

    # Highlight the entire selected row in light red
    if selected_rows:
        column_style += [{
            'if': {'row_index': selected_rows[0]},
            'backgroundColor': 'rgb(255, 220, 220)',
            'border': '1px solid rgb(255, 100, 100)',
        }]
        
    if not is_button:
        return (
            dash.no_update,
            column_style,
            dash.no_update,
            dash.no_update, dash.no_update, dash.no_update,
            dash.no_update, dash.no_update, dash.no_update,
            dash.no_update
        )
    
    # Resets active button list
    if is_reset:
        active_buttons = []
    # Adds triggered button to active buttons list
    else:
        if trigger_id in active_buttons:
            active_buttons = [b for b in active_buttons if b != trigger_id]
        else:
            active_buttons = active_buttons + [trigger_id]

    # Checks active_buttons list to toggle on buttons
    cat_on = 'btn-cat' in active_buttons
    dog_on = 'btn-dog' in active_buttons
    water_on = 'btn-water' in active_buttons
    mountain_wilderness_on = 'btn-mountain_wilderness' in active_buttons
    disaster_tracking_on = 'btn-disaster_tracking' in active_buttons
    
    # Checks rescue filters and adds to list if ON
    active_job_filters = []
    if water_on:
        active_job_filters.append(RESCUE_TYPES['water'])
    if mountain_wilderness_on:
        active_job_filters.append(RESCUE_TYPES['mountain_wilderness'])
    if disaster_tracking_on:
        active_job_filters.append(RESCUE_TYPES['disaster_tracking'])
    
    # Checks species filters and adds to list if ON
    active_species_filters = []
    if cat_on:
        active_species_filters.append('Cat')
    if dog_on:
        active_species_filters.append('Dog')    

    # Compiles query based on active filters
    try:
        if active_job_filters:
            query = {'$or': active_job_filters} if len(active_job_filters) > 1 else active_job_filters[0]
        elif active_species_filters:
            query = {'animal_type': {'$in': active_species_filters}}
        else:
            query = {}

        filtered = pd.DataFrame.from_records(shelter.read(query))
            
        if '_id' in filtered.columns:
            filtered.drop(columns=['_id'], inplace=True)
            
    except Exception as e:
        print(f"Filter attempt failed: {e}. Reverting to previous data.")
        filtered = df.copy()

    return (
        filtered.to_dict('records'),
        column_style,
        '' if is_reset else dash.no_update,
        active_buttons,
        button_on if cat_on else button_off,
        button_on if dog_on else button_off,
        button_on if water_on else button_off,
        button_on if mountain_wilderness_on else button_off,
        button_on if disaster_tracking_on else button_off,
        button_reset,
    )

# This callback updates the piechart's breed distribution based on current
# table data entry selections. It will function only if changes have been made to
# the current filters, nothing happens when a specfic row is selected.
# Breeds where there are less than 1% are now combined into an 'others' category
# to clean up visual display and focus on important data.

@app.callback(
    Output('pie-chart', 'figure'),
    Input('datatable-id', 'derived_virtual_data')
)

##################################################################################
#
# update_pie function
# 
# Description: Builds pie chart and legend from currently selected/filtered data
# table results. Does not change upon row selection, and only updates if filters or
# reset are changed/enacted.
#
# Param: viewData (dict) - Currently visible rows in datatable
#
# Returns: px.pie (dict) - Plotly pie chart object
#
##################################################################################

def update_pie(viewData):
    
    # Validation checks
    if viewData is None or len(viewData) == 0:
        return px.pie(title='No data available')
    
    try:
        dff = pd.DataFrame.from_dict(viewData)
    except Exception as e:
        print(f'Cannot create dataframe: {e}')
        return px.pie(title='No data available')
    
    if dff.empty:
        return px.pie(title='No data available')
    
    # Counts and sorts breeds creating distribution
    if 'breed' in dff.columns:
        breed_counts = dff['breed'].value_counts().reset_index()
        breed_counts.columns = ['breed', 'count']

        # Group breeds under 1% into 'Other'
        total = breed_counts['count'].sum()
        breed_counts['percent'] = breed_counts['count'] / total * 100
        other = breed_counts[breed_counts['percent'] < 1]['count'].sum()
        breed_counts = breed_counts[breed_counts['percent'] >= 1]

        if other > 0:
            other_row = pd.DataFrame([{'breed': 'Other', 'count': other, 'percent': other / total * 100}])
            breed_counts = pd.concat([breed_counts, other_row], ignore_index=True)

        pie_fig = px.pie(
            breed_counts,
            names='breed',
            values='count',
            title='Breed Distribution',
        )
        # Layout configuration, font, background
        pie_fig.update_layout(
            title_x=0.45,
            title_font=dict(size=20, family='Arial Black'),
            paper_bgcolor='#F2EFE9'
        )
        # Hover data formatting
        pie_fig.update_traces(
            hovertemplate='<b>Breed:</b> %{label}<br><b>Count:</b> %{value}<extra></extra>'
        )
        # Early return if all successful
        return pie_fig
    
    return px.pie(title='Breed data unavailable')

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable

@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
           
##############################################################################
# Description: Utilizes selected documents (animal) to read lattitude and
# longitude columns from selected row and creates marker location on map with
# breed and animal name upon selections.
#
# Param: viewData (list of dict): Currently loaded rows in data table.
# Param: index (list of int): Selected row indices.
# 
# Return: list - selected single element of leaflet
#              - empty list if invalid
#
##############################################################################
           
def update_map(viewData, index):
    
    if viewData is None or len(viewData) == 0:
        return []
    
    try:
        dff = pd.DataFrame.from_dict(viewData)
    except Exception as e:
        print(f"Unable to initiate datafram from viewData: {e}")
        return []
    
    # Check for empty dataframe post conversion
    if dff.empty:
        return []
    
    # Check sufficient columns are available for leaflet render
    if len(dff.columns) <= max(REQUIRED_COLUMNS.values()):
        print("Insufficient columns to render map.")
        return []
    
# Because we only allow single row selection, the list can 
# be converted to a row index here
    
    if index is None or len(index) == 0:
        row = 0
    else: 
        row = index[0]
        
    if row >= len(dff):
        row = 0

# Austin TX is at [30.75,-97.48]

    try:
        lat = float(dff.loc[dff.index[row], 'location_lat'])
        lon = float(dff.loc[dff.index[row], 'location_long'])
    except (ValueError, TypeError) as e:
        print(f"Invalid coordinates row {row}: {e}. Default to Austin, TX.")
        lat, lon = 30.75, -97.48
        
    animal_name = dff.loc[dff.index[row], 'name'] if 'name' in dff.columns else ''
    breed_label = dff.loc[dff.index[row], 'breed'] if 'breed' in dff.columns else 'Unknown'
    
    display_name = (
        animal_name
        if pd.notna(animal_name) and str(animal_name).strip() != ''
        else 'No Name'
    )
    
    map_component = []
    
    try:
        map_component = [
            dl.Map(
                style={'width': '100%', 'height': '500px'},
                center=[lat, lon],
                zoom=10,
                children=[
                    dl.TileLayer(id='base-layer-id'),
                    dl.Marker(
                        position=[lat, lon],
                        children=[
                            dl.Tooltip(breed_label),
                            dl.Popup([
                                html.H1('Animal Name:', style={
                                    'textAlign': 'center',
                                    'fontSize': '16px',
                                    'marginBottom': '5px'
                                }),
                                html.P(display_name, style={
                                    'textAlign': 'center',
                                    'fontSize': '14px',
                                    'fontWeight': 'bold'
                                })
                            ])
                        ]
                    )
                ]
            )
        ]
        
    except Exception as e:
        print(f"Map did not render properly: {e}")
        
    return map_component

# Run app and display result in jupyterlab mode, note, 
# if you have previously run a prior app, the default port of 8050 may not be available,
# if so, try setting an alternate port.
           
app.run_server()

INFO:CRUD_Python_Module:Connection Successful
INFO:CRUD_Python_Module:Read request returned 10012 documents matching query
INFO:CRUD_Python_Module:Read request returned 10012 documents matching query


Dash app running on https://mysterydivide-contactpixel-3000.codio.io/proxy/8050/


INFO:CRUD_Python_Module:Read request returned 17 documents matching query
INFO:CRUD_Python_Module:Read request returned 10012 documents matching query
INFO:CRUD_Python_Module:Read request returned 5 documents matching query
INFO:CRUD_Python_Module:Read request returned 10012 documents matching query
INFO:CRUD_Python_Module:Read request returned 4 documents matching query
INFO:CRUD_Python_Module:Read request returned 10012 documents matching query
